In [2]:
import random
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass
import numpy as np


@dataclass
class Book:
    book_id: int
    category: str
    popularity: float
    available: bool
class ParallelCellularLibraryOptimizer:
    def __init__(self, rows: int, cols: int, books: list[Book], generations: int = 100):
        self.rows = rows
        self.cols = cols
        self.generations = generations


        total_slots = rows * cols
        if len(books) > total_slots:
            raise ValueError("More books than available shelf slots.")
        self.items = books + [None] * (total_slots - len(books))

        random.shuffle(self.items)
        self.grid = np.array(self.items, dtype=object).reshape(rows, cols)


        mid_row = (rows - 1) / 2.0
        self.row_accessibility = [
            1.0 - (abs(r - mid_row) / max(1.0, mid_row)) * 0.7 for r in range(rows)
        ]

    def get_neighbors(self, r: int, c: int) -> list[tuple[int, int]]:
        """Return valid Moore neighborhood coordinates."""
        neighbors = []
        for dr in (-1, 0, 1):
            for dc in (-1, 0, 1):
                if dr == 0 and dc == 0:
                    continue
                nr, nc = r + dr, c + dc
                if 0 <= nr < self.rows and 0 <= nc < self.cols:
                    neighbors.append((nr, nc))
        return neighbors

    def cell_fitness(self, r: int, c: int, grid: np.ndarray) -> float:
        """Evaluate local fitness for a single cell based on grouping and accessibility."""
        book = grid[r, c]
        if book is None:
            return 0.0


        effective_demand = book.popularity * (1.0 if book.available else 0.2)
        accessibility_score = effective_demand * self.row_accessibility[r] * 10.0


        neighbors = self.get_neighbors(r, c)
        same_category_count = 0
        for nr, nc in neighbors:
            neighbor_book = grid[nr, nc]
            if neighbor_book is not None and neighbor_book.category == book.category:
                same_category_count += 1
        grouping_score = same_category_count * 2.5

        return accessibility_score + grouping_score

    def total_fitness(self, grid: np.ndarray) -> float:
        """Calculate overall fitness across the entire library grid."""
        return sum(
            self.cell_fitness(r, c, grid)
            for r in range(self.rows)
            for c in range(self.cols)
        )

    def _propose_swap(self, coords: tuple[int, int]) -> tuple[tuple[int, int], tuple[int, int], float]:
        """Parallel worker: finds the best neighbor swap for cell (r, c)."""
        r, c = coords
        neighbors = self.get_neighbors(r, c)
        best_target = None
        best_gain = 0.0

        current_local = self.cell_fitness(r, c, self.grid)

        for nr, nc in neighbors:
            if self.grid[r, c] is None and self.grid[nr, nc] is None:
                continue

            neighbor_local = self.cell_fitness(nr, nc, self.grid)
            before_score = current_local + neighbor_local

            self.grid[r, c], self.grid[nr, nc] = self.grid[nr, nc], self.grid[r, c]
            after_score = self.cell_fitness(r, c, self.grid) + self.cell_fitness(nr, nc, self.grid)
            self.grid[r, c], self.grid[nr, nc] = self.grid[nr, nc], self.grid[r, c]

            gain = after_score - before_score
            if gain > best_gain:
                best_gain = gain
                best_target = (nr, nc)

        return (r, c), best_target, best_gain

    def optimize(self) -> tuple[np.ndarray, list[float]]:
        """Run parallel cellular evolution across generations."""
        history = [self.total_fitness(self.grid)]
        all_coords = [(r, c) for r in range(self.rows) for c in range(self.cols)]

        with ThreadPoolExecutor() as executor:
            for gen in range(self.generations):
                proposals = list(executor.map(self._propose_swap, all_coords))

                proposals.sort(key=lambda x: x[2], reverse=True)

                locked_cells = set()
                swaps_made = 0

                for src, dst, gain in proposals:
                    if dst is not None and gain > 0:
                        if src not in locked_cells and dst not in locked_cells:
                            self.grid[src], self.grid[dst] = self.grid[dst], self.grid[src]
                            locked_cells.add(src)
                            locked_cells.add(dst)
                            swaps_made += 1

                if swaps_made == 0 and gen < self.generations - 1:
                    c1, c2 = random.sample(all_coords, 2)
                    self.grid[c1], self.grid[c2] = self.grid[c2], self.grid[c1]

                history.append(self.total_fitness(self.grid))

        return self.grid, history


def display_shelf(grid: np.ndarray, title: str):
    """Print a visual representation of the library grid."""
    print(f"\n=== {title} ===")
    for r in range(grid.shape[0]):
        row_str = []
        for c in range(grid.shape[1]):
            book = grid[r, c]
            if book is None:
                row_str.append("[  EMPTY  ]")
            else:
                avail_tag = "+" if book.available else "-"
                row_str.append(f"[{book.category[:3].upper()}:{book.popularity:.1f}{avail_tag}]")
        print(f"Row {r} (Shelf): " + " ".join(row_str))


if __name__ == "__main__":
    random.seed(42)
    categories = ["Fiction", "Science", "History", "Art"]

    sample_books = [
        Book(
            book_id=i,
            category=random.choice(categories),
            popularity=round(random.uniform(0.1, 1.0), 1),
            available=random.choice([True, True, False]),
        )
        for i in range(22)
    ]

    optimizer = ParallelCellularLibraryOptimizer(
        rows=4, cols=6, books=sample_books, generations=50
    )

    display_shelf(optimizer.grid, "Initial Random Shelf Arrangement")
    initial_score = optimizer.total_fitness(optimizer.grid)
    print(f"Initial Fitness Score: {initial_score:.2f}")

    optimized_grid, fitness_history = optimizer.optimize()

    display_shelf(optimized_grid, "Optimized Shelf Arrangement")
    final_score = fitness_history[-1]
    print(f"Final Fitness Score:   {final_score:.2f} (+{final_score - initial_score:.2f})")



=== Initial Random Shelf Arrangement ===
Row 0 (Shelf): [HIS:0.6-] [FIC:0.2+] [FIC:0.4+] [FIC:0.1+] [SCI:0.3-] [HIS:0.2+]
Row 1 (Shelf): [FIC:0.7-] [ART:0.3-] [FIC:0.6+] [HIS:0.9-] [FIC:0.6+] [FIC:0.1+]
Row 2 (Shelf): [SCI:0.7-] [HIS:0.2+] [ART:0.6+] [FIC:0.2+] [  EMPTY  ] [  EMPTY  ]
Row 3 (Shelf): [SCI:0.7+] [HIS:0.8-] [FIC:0.6-] [FIC:0.6-] [HIS:0.8+] [HIS:1.0+]
Initial Fitness Score: 118.56

=== Optimized Shelf Arrangement ===
Row 0 (Shelf): [HIS:0.6-] [SCI:0.7+] [SCI:0.7-] [  EMPTY  ] [FIC:0.1+] [ART:0.6+]
Row 1 (Shelf): [HIS:0.9-] [HIS:1.0+] [FIC:0.7-] [FIC:0.2+] [FIC:0.6+] [FIC:0.6-]
Row 2 (Shelf): [HIS:0.2+] [HIS:0.2+] [HIS:0.8+] [FIC:0.2+] [FIC:0.4+] [FIC:0.6-]
Row 3 (Shelf): [ART:0.3-] [SCI:0.3-] [  EMPTY  ] [FIC:0.6+] [FIC:0.1+] [HIS:0.8-]
Final Fitness Score:   204.81 (+86.25)
